In [34]:
# Import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn imports
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LogisticRegression, LassoCV, ElasticNetCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif, VarianceThreshold, RFE, SelectFromModel, mutual_info_classif
from sklearn.metrics import (confusion_matrix, classification_report, roc_auc_score, 
                             roc_curve, precision_recall_curve, f1_score, 
                             precision_score, recall_score, average_precision_score)
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier

# Set random seed for reproducibility
np.random.seed(42)

print("All libraries imported successfully!")

All libraries imported successfully!


In [35]:
# Load the ASD multi-omics dataset
DATA_DIR = "../data/ASD_dataset/"

print("Loading ASD multi-omics dataset...\n")

# Load covariates (includes target variable 'ASD')
covariates = pd.read_csv(f"{DATA_DIR}ASD_covariates.csv")
print(f"✓ Covariates: {covariates.shape[0]} samples, {covariates.shape[1]} variables")

# Load genotype data (SNPs)
genotypes = pd.read_csv(f"{DATA_DIR}ASD_genotypes.csv")
print(f"✓ Genotypes: {genotypes.shape[0]} samples, {genotypes.shape[1]-1} SNPs")

# Load gene expression data (using expanded 100-gene dataset)
expression = pd.read_csv(f"{DATA_DIR}ASD_expression.csv")
print(f"✓ Expression: {expression.shape[0]} samples, {expression.shape[1]-1} genes")

# Set sample ID as index for easy merging
covariates_idx = covariates.set_index('sample')
genotypes_idx = genotypes.set_index('sample')
expression_idx = expression.set_index('sample')

# Create integrated feature matrix (multi-omics integration)
X_integrated = pd.DataFrame(index=expression_idx.index)

# Add gene expression features (NO prefix - use gene names directly)
for gene in expression_idx.columns:
    X_integrated[gene] = expression_idx[gene]

# Add top 10 SNP features (prefix with geno_)
for snp in genotypes_idx.columns[:10]:
    X_integrated[f'geno_{snp}'] = genotypes_idx[snp]

# Add clinical covariates
X_integrated['Age'] = covariates_idx['Age']
X_integrated['Sex'] = covariates_idx['Sex']

# Target variable: ASD status (1=case, 0=control)
y = covariates_idx['ASD'].values

print(f"\n✓ Integrated feature matrix: {X_integrated.shape}")
print(f"✓ Target distribution: {y.sum()} ASD cases, {len(y)-y.sum()} controls")
print(f"  Class ratio: {y.mean():.1%} cases")

X_train, X_test, y_train, y_test = train_test_split(
    X_integrated, y,
    test_size=0.3,      # 30% for testing
    stratify=y,         # Preserve class proportions
    random_state=42
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # Use same scaling

# Train a simple model for demonstration
rf_model = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)
rf_model.fit(X_train_scaled, y_train)

Loading ASD multi-omics dataset...

✓ Covariates: 300 samples, 29 variables
✓ Genotypes: 300 samples, 50 SNPs
✓ Expression: 300 samples, 100 genes

✓ Integrated feature matrix: (300, 112)
✓ Target distribution: 200 ASD cases, 100 controls
  Class ratio: 66.7% cases


RandomForestClassifier(max_depth=5, n_estimators=50, random_state=42)

# Feature Engineering Methods

Demonstrates 7+ methods:
1. Variance Threshold
2. SelectKBest (Univariate)
3. Recursive Feature Elimination (RFE)
4. L1 Regularization (Lasso)
5. Tree-based Selection
6. Mutual Information
7. Boruta Algorithm

In [36]:
# Variance Threshold
print("=" * 80)
print("METHOD 1: VARIANCE THRESHOLD")
print("=" * 80)

# Remove features with low variance
var_selector = VarianceThreshold(threshold=0.1)
var_selector.fit(X_train)

# Get selected features
selected_var = X_integrated.columns[var_selector.get_support()].tolist()

print(f"\nOriginal features: {X_integrated.shape[1]}")
print(f"After variance threshold: {len(selected_var)}")
print(f"Removed: {X_integrated.shape[1] - len(selected_var)} low-variance features")

print("\n✓ Fast preprocessing step - removes technical noise")
print("  Goal: Remove constants and near-constants")

METHOD 1: VARIANCE THRESHOLD

Original features: 112
After variance threshold: 108
Removed: 4 low-variance features

✓ Fast preprocessing step - removes technical noise
  Goal: Remove constants and near-constants


In [37]:
# SelectKBest (Univariate Statistical Tests)
print("=" * 80)
print("METHOD 2: SELECTKBEST (UNIVARIATE F-TEST)")
print("=" * 80)

# Select top 10 features by ANOVA F-test
selector_kbest = SelectKBest(f_classif, k=10)
selector_kbest.fit(X_train, y_train)

# Get selected features and their scores
selected_kbest = X_integrated.columns[selector_kbest.get_support()].tolist()
scores_kbest = selector_kbest.scores_

# Create DataFrame with scores
kbest_df = pd.DataFrame({
    'Feature': X_integrated.columns,
    'F_Score': scores_kbest,
    'Selected': selector_kbest.get_support()
}).sort_values('F_Score', ascending=False)

print("\nTop 10 Selected Features:")
print(kbest_df.head(10)[['Feature', 'F_Score', 'Selected']].to_string(index=False))

print("\n✓ Fast statistical filter - each feature tested independently")
print("  Good for: 20,000 → 1,000 features")
print("  Con: Misses gene interactions")

METHOD 2: SELECTKBEST (UNIVARIATE F-TEST)

Top 10 Selected Features:
   Feature   F_Score  Selected
     SCN2A 43.024780      True
      CHD8 28.714485      True
    SHANK3 28.102409      True
       Sex 17.082435      True
      ADNP  4.440548      True
   CACNA1B  3.254118      True
geno_snp_9  2.948046      True
     CREB1  2.904905      True
      NTF3  2.660353      True
geno_snp_3  2.533433      True

✓ Fast statistical filter - each feature tested independently
  Good for: 20,000 → 1,000 features
  Con: Misses gene interactions


# EXplainable AI: Feature Importance for Biomarker Discovery

Demonstrates 7+ methods and translating importance to clinical biomarkers:
1. Tree-based importance
2. Permutation importance
3. Linear coefficients
4. SHAP values
5. Drop-column importance
6. Comparison across methods
7. Biomarker panel validation

In [38]:
# Recursive Feature Elimination (RFE)
print("=" * 80)
print("METHOD 3: RECURSIVE FEATURE ELIMINATION (RFE)")
print("=" * 80)

# Use RFE to select top 10 features
estimator = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)
selector_rfe = RFE(estimator, n_features_to_select=10, step=2)

print("\nRunning RFE (this takes a moment...)\n")
selector_rfe.fit(X_train_scaled, y_train)

# Get selected features and their ranking
selected_rfe = X_integrated.columns[selector_rfe.get_support()].tolist()
ranking_rfe = selector_rfe.ranking_

rfe_df = pd.DataFrame({
    'Feature': X_integrated.columns,
    'Ranking': ranking_rfe,
    'Selected': selector_rfe.get_support()
}).sort_values('Ranking')

print("Selected Features (Ranking=1):")
print(rfe_df[rfe_df['Ranking']==1]['Feature'].tolist())

print("\n✓ Wrapper method - optimizes feature combinations")
print("  Good for: 1,000 → 20 biomarker panel")
print("  Con: Computationally expensive")

METHOD 3: RECURSIVE FEATURE ELIMINATION (RFE)

Running RFE (this takes a moment...)

Selected Features (Ranking=1):
['GABBR2', 'SCN3A', 'GRIN2B', 'SCN2A', 'CDH2', 'CHD8', 'EHMT1', 'ARID1B', 'SHANK3', 'NTRK1']

✓ Wrapper method - optimizes feature combinations
  Good for: 1,000 → 20 biomarker panel
  Con: Computationally expensive


In [39]:
# L1 Regularization (Lasso/ElasticNet)
print("=" * 80)
print("METHOD 4: L1 REGULARIZATION (ELASTICNET)")
print("=" * 80)

# Use ElasticNet (L1 + L2) for feature selection
# l1_ratio=0.9 means 90% L1, 10% L2
elastic = ElasticNetCV(l1_ratio=0.9, cv=5, random_state=42, max_iter=2000)
elastic.fit(X_train_scaled, y_train)

# Features with non-zero coefficients are selected
selected_lasso = X_integrated.columns[elastic.coef_ != 0].tolist()

lasso_df = pd.DataFrame({
    'Feature': X_integrated.columns,
    'Coefficient': elastic.coef_,
    'Selected': elastic.coef_ != 0
}).sort_values('Coefficient', key=abs, ascending=False)

print(f"\nSelected {len(selected_lasso)} features with non-zero coefficients:")
print(lasso_df[lasso_df['Selected']][['Feature', 'Coefficient']].to_string(index=False))

print("\n✓ Embedded method - automatic sparse selection")
print("  ElasticNet good for correlated genes (genomics)")
print("  Results in sparse models - perfect for biomarker panels")

METHOD 4: L1 REGULARIZATION (ELASTICNET)

Selected 14 features with non-zero coefficients:
   Feature  Coefficient
      CHD8     0.284540
     SCN2A    -0.206205
    SHANK3    -0.151685
       Sex     0.040558
      CDH2     0.018756
geno_snp_4    -0.017012
     NPAS4     0.014986
geno_snp_1     0.010954
geno_snp_6    -0.007308
    GABRA2     0.004784
geno_snp_9    -0.004412
    SLC6A3     0.002896
    ARID1B     0.002028
    SLC6A4     0.001385

✓ Embedded method - automatic sparse selection
  ElasticNet good for correlated genes (genomics)
  Results in sparse models - perfect for biomarker panels


In [40]:
# Tree-Based Feature Selection
print("=" * 80)
print("METHOD 5: TREE-BASED SELECTION")
print("=" * 80)

# Use SelectFromModel with Random Forest
selector_tree = SelectFromModel(
    RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    threshold='median'  # Keep features above median importance
)

selector_tree.fit(X_train_scaled, y_train)
selected_tree_sel = X_integrated.columns[selector_tree.get_support()].tolist()

print(f"\nSelected {len(selected_tree_sel)} features above median importance:")
print(', '.join(selected_tree_sel))

print("\n✓ Fast embedded method using tree importances")
print("  Captures non-linear relationships and interactions")

METHOD 5: TREE-BASED SELECTION

Selected 56 features above median importance:
MECP2, GRIN2B, SCN2A, NRXN1, CHD8, SYNGAP1, ARID1B, SHANK3, FOXP1, DYRK1A, ANK2, KMT2E, CTNNB1, DEAF1, NLGN4X, TSC2, NF1, EHMT1, MBD5, TBR1, SYN2, STX1A, SYT1, CACNA1A, SCN3A, SCN8A, GABRA1, GABRA2, GABRA5, GABRG2, GABBR1, GABBR2, NEUROD1, NEUROD2, NEUROG1, MAP2, MAPT, NCAM1, NCAM2, CDH2, CTNND2, NPAS4, CAMK2B, PRKCA, PRKCB, GRM1, HTR2A, DRD1, DRD2, SLC6A4, SLC1A1, BDNF, NTF3, NTRK1, CREB1, Sex

✓ Fast embedded method using tree importances
  Captures non-linear relationships and interactions


In [41]:
# Mutual Information
print("=" * 80)
print("METHOD 6: MUTUAL INFORMATION")
print("=" * 80)

# Calculate mutual information scores
mi_scores = mutual_info_classif(X_train, y_train, random_state=42)

mi_df = pd.DataFrame({
    'Feature': X_integrated.columns,
    'MI_Score': mi_scores
}).sort_values('MI_Score', ascending=False)

# Select top 10
selected_mi = mi_df.head(10)['Feature'].tolist()

print("\nTop 10 Features by Mutual Information:")
print(mi_df.head(10).to_string(index=False))

print("\n✓ Captures non-linear dependencies")
print("  Good for complex biological relationships")
print("  MI = how much knowing gene tells you about disease")

METHOD 6: MUTUAL INFORMATION

Top 10 Features by Mutual Information:
Feature  MI_Score
   CHD8  0.125059
  SCN2A  0.092217
  GRIA2  0.081513
   POGZ  0.081431
CACNA1B  0.073665
   PCLO  0.073535
  HTR2A  0.067072
 SHANK3  0.064914
   BDNF  0.053862
  PRKCA  0.043998

✓ Captures non-linear dependencies
  Good for complex biological relationships
  MI = how much knowing gene tells you about disease


In [42]:
# Boruta Algorithm (if available)
print("=" * 80)
print("METHOD 7: BORUTA ALGORITHM")
print("=" * 80)

# Goal:
#   Identify ALL features that carry information about the target,
#   not just a minimal predictive subset.
#
# Key idea of Boruta:
#   1. Create "shadow features" by randomly permuting each real feature. Same values. 
#      Same distribution (mean, variance). Broken association with the target.
#      x = [10, 12, 11, 13, 14], shadow(x) = [13, 10, 14, 11, 12]
#   2. Train a Random Forest on real + shadow features.
#   3. Compute feature importance (mean decrease in impurity).
#   4. A real feature is:
#        - CONFIRMED if it consistently beats the best shadow feature
#        - REJECTED if it consistently performs worse
#        - TENTATIVE if the evidence is insufficient

# Out-of-bag (OOB) error is an internal Random Forest estimate of generalization 
# performance computed by predicting each training sample using only the trees 
# that did not see that sample during bootstrap training.
from boruta import BorutaPy
from sklearn.base import clone

class BorutaWithOOB(BorutaPy):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.oob_scores_ = []

    def _get_imp(self, X, y):
        """
        This method is called once per Boruta iteration.
        It fits the Random Forest and computes feature importances.
        """
        estimator = clone(self.estimator)
        estimator.fit(X, y)

        # Safely log OOB score if available
        if hasattr(estimator, "oob_score_"):
            self.oob_scores_.append(estimator.oob_score_)

        return estimator.feature_importances_

try:
    from boruta import BorutaPy
    
    # Initialize Boruta
    rf_boruta = RandomForestClassifier(n_estimators=50, bootstrap=True,
                                       oob_score=True, max_depth=5, random_state=42)
    boruta = BorutaWithOOB(rf_boruta, n_estimators='auto', random_state=42, max_iter=20)
    
    print("\nRunning Boruta (this may take a minute...)\n")
    boruta.fit(X_train.values, y_train)
    
    # Get selected features
    selected_boruta = X_integrated.columns[boruta.support_].tolist()
    tentative_boruta = X_integrated.columns[boruta.support_weak_].tolist()
    
    print("Mean OOB score during Boruta:", sum(boruta.oob_scores_) / len(boruta.oob_scores_))
    
    print(f"Selected (confirmed): {len(selected_boruta)} features")
    print(', '.join(selected_boruta))
    
    print(f"\nTentative: {len(tentative_boruta)} features")
    print(', '.join(tentative_boruta) if tentative_boruta else 'None')
    
    print("\n✓ All-relevant selection - finds ALL important features")
    print("  Compares real features to random shadows")
    print("  Statistically grounded")
    
except ImportError:
    print("\n⚠️  Boruta not installed.")
    print("Install with: pip install boruta")
    print("\nSkipping Boruta demonstration...")

METHOD 7: BORUTA ALGORITHM

Running Boruta (this may take a minute...)

Mean OOB score during Boruta: 0.742857142857143
Selected (confirmed): 3 features
SCN2A, CHD8, SHANK3

Tentative: 0 features
None

✓ All-relevant selection - finds ALL important features
  Compares real features to random shadows
  Statistically grounded


In [43]:
# Compare Feature Selection Methods
print("=" * 80)
print("FEATURE SELECTION METHOD COMPARISON")
print("=" * 80)

# Compare top features from different methods
all_methods = {
    'SelectKBest': set(selected_kbest),
    'RFE': set(selected_rfe),
    'Lasso': set(selected_lasso),
    'Tree-based': set(selected_tree_sel),
    'Mutual Info': set(selected_mi),
    'Boruta': set(selected_boruta)
}

print("\nNumber of features selected by each method:")
for method, features in all_methods.items():
    print(f"  {method:15s}: {len(features):2d} features")

# Find consensus features (selected by multiple methods)
from collections import Counter
all_selected = []
for features in all_methods.values():
    all_selected.extend(features)

feature_counts = Counter(all_selected)
consensus_features = {f: c for f, c in feature_counts.items() if c >= 3}

print("\nConsensus Features (selected by 3+ methods):")
for feature, count in sorted(consensus_features.items(), key=lambda x: x[1], reverse=True):
    print(f"  {feature:20s}: selected by {count}/5 methods")

print("\n✓ Features selected by multiple methods are most robust!")
print("  These are your best biomarker candidates")

FEATURE SELECTION METHOD COMPARISON

Number of features selected by each method:
  SelectKBest    : 10 features
  RFE            : 10 features
  Lasso          : 14 features
  Tree-based     : 56 features
  Mutual Info    : 10 features
  Boruta         :  3 features

Consensus Features (selected by 3+ methods):
  SHANK3              : selected by 6/5 methods
  SCN2A               : selected by 6/5 methods
  CHD8                : selected by 6/5 methods
  Sex                 : selected by 3/5 methods
  CDH2                : selected by 3/5 methods
  ARID1B              : selected by 3/5 methods

✓ Features selected by multiple methods are most robust!
  These are your best biomarker candidates
